# OpenICU-YAIB validation: mimic_demo

Build an OpenICU YAIB-style dynamic wide table and optionally compare it with an R/`ricu` export.

The R reference uses the built-in `ricu` source `mimic_demo`.

This notebook assumes that the OpenICU concept parquets already contain `stay_id`, integer-hour `time`, and `numeric_value`. Therefore `ICUSTAYS_CSV=None` and `INCLUDE_GRID=False` are the defaults. Set an ICU-stay CSV and enable the grid only when the stay table has the normalized columns `subject_id`, `hadm_id`, `stay_id`, `intime`, and `outtime`.


In [ ]:
from pathlib import Path

from openicu_yaib import build_and_write_yaib_wide_for_dataset

DATASET = "mimic_demo"
RICU_SRC = "mimic_demo"
OPENICU_CONCEPT_VERSION = "1.0.0"
MAX_HOURS = 7 * 24

# Keep every dataset in its own output directory.
OUTPUT_ROOT = Path.home() / "output" / "openicu_yaib" / DATASET

# Override these paths for the local OpenICU/RICU layout.
CONCEPT_ROOT = None
RICU_CONCEPT_DICT = None
ICUSTAYS_CSV = None
INCLUDE_GRID = False

# "warn" is useful during initial cross-dataset coverage checks.
MISSING_CONCEPTS = "warn"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {DATASET}")
print(f"Output:  {OUTPUT_ROOT}")


In [ ]:
# Export all available hours for downstream analysis.
all_hours = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=None,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    include_grid=INCLUDE_GRID,
    missing_concepts=MISSING_CONCEPTS,
)

print(f"Wrote: {all_hours.output_path}")
all_hours.summary


In [ ]:
# Export the first seven days for comparison with R/ricu.
one_week = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    include_grid=INCLUDE_GRID,
    missing_concepts=MISSING_CONCEPTS,
)

print(f"Wrote: {one_week.output_path}")
one_week.summary


## Export the R/`ricu` reference

From the repository root, run:

```bash
RICU_OUT_DIR="$HOME/output/openicu_yaib/mimic_demo" \
Rscript scripts/datasets/export_ricu_mimic_demo.R
```

This runs both the dynamic-variable and stay-window exports. The R reference uses the built-in `ricu` source `mimic_demo`.


In [ ]:
from openicu_yaib import (
    compare_openicu_wide_to_ricu_for_dataset,
    display_comparison_overview,
)

ricu_dynamic = OUTPUT_ROOT / f"ricu_dynamic_vars_{RICU_SRC}.parquet"
ricu_windows = OUTPUT_ROOT / f"ricu_stay_windows_{RICU_SRC}.parquet"

if ricu_dynamic.is_file() and ricu_windows.is_file():
    comparison = compare_openicu_wide_to_ricu_for_dataset(
        dataset=DATASET,
        max_hours=MAX_HOURS,
        output_root=OUTPUT_ROOT,
        openicu_wide_path=one_week.output_path,
        ricu_dynamic_path=ricu_dynamic,
        ricu_stay_windows_path=ricu_windows,
    )
    print(f"Wrote reports to: {comparison.reports_dir}")
    comparison_tables = display_comparison_overview(comparison)
    comparison_tables
else:
    print("RICU reference files are missing; run the dataset-specific R script first.")
    print(ricu_dynamic)
    print(ricu_windows)
